# Agent Loop

### In this exersie we visualize the checklist that an Agent goes through, when it is going through it's loop with tool_calls before arriving at the final text response.
### To achive this, we create two main funcitons: create_checklist, mark_checklist; and a helper function: get_checklist_report.

In [142]:
from rich.console import Console
from openai import OpenAI
from dotenv import load_dotenv
import json
load_dotenv(override=True)

True

In [143]:
checklist = []
completions = []

In [144]:
# This function uses Console() to print formatted text in the terminal

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [145]:
# This funciton analyses the completion status of the checklist items. Then shows, and returns that completion report though the 'result' variable

def get_checklist_report()-> str:
    result = ""
    for index, item in enumerate(checklist):
        if completions[index]==True:
            result+=f"Checklist #{index+1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result+=f"Checklist #{index+1}: {item}\n" 
    show(result)
    return(result)

In [146]:
# This will be a tool for the llm to create a checklist, to achive its goal

def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completions.extend([False]*len(descriptions))
    return get_checklist_report()

In [147]:
# Test create_checklist
checklist, completions = [], []
create_checklist(["Eat bananas", "Go for a run", "Sleep for 8 hours"])

Checklist #1: Eat bananas
Checklist #2: Go for a run
Checklist #3: Sleep for 8 hours

'Checklist #1: Eat bananas\nChecklist #2: Go for a run\nChecklist #3: Sleep for 8 hours\n'

In [148]:
# This will be a tool for the llm to mark the completed task as True, when completed

def mark_completion(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(completions):
        completions[index-1] = True
    else:
        print(f"The index {index} does not exist\n")
    show(completion_notes)
    return get_checklist_report()

In [149]:
mark_completion(1, "Ate two bananas")

Ate two bananas

Checklist #1: Eat bananas
Checklist #2: Go for a run
Checklist #3: Sleep for 8 hours

'Checklist #1: [green][strike]Eat bananas[/strike][/green]\nChecklist #2: Go for a run\nChecklist #3: Sleep for 8 hours\n'

In [150]:

create_checklist_json = {
    "name": "create_checklist",
    "description": "use this tool to create a checklist for your plan of action",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type':'array', 
                'items': {'type': 'string'},
                #'description': 'An array of descriptions for checklist. Each description is considered a checklist item'
                'title': 'Descriptions of checklist items'
                }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

'''
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}
'''

mark_completion_json = {
    "name": "mark_completion",
    "description": "use this tool to mark the completion of a checklist item",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type":"integer",
                "description": "The index of the checklist item that is completed. The range of index is 1 <= index <= number of checklist items"
                },
            "completion_notes": {
                "type": "string",
                "description": "A short notes about the completion of the checklist item"
            }
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": create_checklist_json},
    {"type": "function", "function": mark_completion_json}
]

tool_list = {
    "create_checklist": create_checklist,
    "mark_completion": mark_completion
    }

# test tool_list
checklist, completions = [],[]
tool = tool_list["create_checklist"]
tool(["Open the laptop", "log in", "Finish drv-123 ticket"])

Checklist #1: Open the laptop
Checklist #2: log in
Checklist #3: Finish drv-123 ticket

'Checklist #1: Open the laptop\nChecklist #2: log in\nChecklist #3: Finish drv-123 ticket\n'

In [151]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        tool = tool_list[tool_name]
        
        tool_result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(tool_result), "tool_call_id": tool_call.id})

    return results

In [152]:
# Initiate an openai client
openai = OpenAI()

In [156]:
system_prompt = """
You are given a question to solve, using the checklist tools, to setup a checklist, finish and mark each step, and reply with a solution.
If the question is missing a quantity, do not ask the user for clarification. Instead, make that a step, and make a reasonable assumption.
Provide your solution in Rich console markup without code blocks.
""".strip()

user_message = """
A company’s revenue increased by 20% in the first year and then decreased by 10% in the second year. If the revenue after the two years was $1,080,000, what was the company's revenue at the beginning of the first year?

A. $900,000
 B. $950,000
 C. $1,000,000
 D. $1,050,000
 E. $1,100,000
""".strip()

In [157]:
def agent_loop(user_message):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}]
    response = openai.chat.completions.create(model= "gpt-5.4-mini", messages = messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model= "gpt-5.4-mini", messages = messages, tools=tools)
    show(response.choices[0].message.content)

In [158]:
checklist, completions = [],[]
agent_loop(user_message)

Checklist #1: Determine the combined multiplier for a 20% increase followed by a 10% decrease.
Checklist #2: Work backward from the final revenue of $1,080,000 to find the initial revenue.
Checklist #3: Match the result to the provided multiple-choice options and present the answer.

A 20% increase multiplies revenue by 1.2; a 10% decrease multiplies by 0.9, for a net multiplier of 1.08.

Checklist #1: Determine the combined multiplier for a 20% increase followed by a 10% decrease.
Checklist #2: Work backward from the final revenue of $1,080,000 to find the initial revenue.
Checklist #3: Match the result to the provided multiple-choice options and present the answer.

Initial revenue = 1,080,000 ÷ 1.08 = 1,000,000.

Checklist #1: Determine the combined multiplier for a 20% increase followed by a 10% decrease.
Checklist #2: Work backward from the final revenue of $1,080,000 to find the initial revenue.
Checklist #3: Match the result to the provided multiple-choice options and present the answer.

$1,000,000 corresponds to option C.

Checklist #1: Determine the combined multiplier for a 20% increase followed by a 10% decrease.
Checklist #2: Work backward from the final revenue of $1,080,000 to find the initial revenue.
Checklist #3: Match the result to the provided multiple-choice options and present the answer.

The revenue changed by a net multiplier of:

1.20 × 0.90 = 1.08

So the original revenue was:

$1,080,000 ÷ 1.08 = $1,000,000

Answer: C. $1,000,000